# tiles-to-mesh — Quickstart

This notebook demonstrates the full workflow:
1. **Select** a geographic region using an interactive map
2. **Fetch** 3D tiles from Google's Map Tiles API
3. **Clean up** the mesh using Blender operations
4. **Preview** the result in an interactive 3D viewer
5. **Export** to GLB format

In [ ]:
# ── Install (run this cell first) ──────────────────────────────────────
# Colab: clone the repo and install in pure-Python mode (no Rust needed)
import os, sys, subprocess

if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiles-to-mesh-plugin"):
        subprocess.run(["git", "clone", "https://github.com/byeSystem32/tiles-to-mesh-plugin.git",
                        "/content/tiles-to-mesh-plugin"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "ipywidgets", "pythreejs", "tqdm", "requests", "Pillow"], check=True)
    if "/content/tiles-to-mesh-plugin/python" not in sys.path:
        sys.path.insert(0, "/content/tiles-to-mesh-plugin/python")

# Local Jupyter: just ensure `maturin develop --release` was run,
# or `pip install -e .` from the repo root.

import tiles_to_mesh as ttm
print(f"tiles-to-mesh v{ttm.__version__} loaded")

# ── Your Google Maps API key (must have Map Tiles API enabled) ────────
API_KEY = "YOUR_API_KEY_HERE"

## Step 1: Select a Region

Draw a polygon on the map to define your area of interest.

In [ ]:
# Interactive map selector
# In Colab this automatically uses a Colab-native implementation.
# Draw a polygon on the map, and the coordinates are sent to Python.
selector = ttm.MapSelector(
    api_key=API_KEY,
    center=(40.748817, -73.985428),  # NYC - Empire State Building
    zoom=17,
    map_type="satellite",
)
selector.show()

# ── Alternative: set coordinates programmatically (no map needed) ─────
# selector.set_region_programmatic([
#     (40.7484, -73.9867),
#     (40.7492, -73.9867),
#     (40.7492, -73.9845),
#     (40.7484, -73.9845),
# ], name="Empire State Building")
#
# ── Or create a Region directly ───────────────────────────────────────
# region = ttm.Region.from_bounds(
#     south=40.7484, west=-73.9867,
#     north=40.7492, east=-73.9845,
#     name="Empire State Building"
# )

## Step 2: Fetch 3D Mesh

Download and assemble 3D tiles for the selected region.

In [ ]:
mesh = ttm.fetch_mesh(
    selector.region,
    api_key=API_KEY,
    lod=3,                    # 1 (coarse) to 5 (max detail)
    mode="photorealistic",    # or "geometry" for untextured
)

mesh.info()

## Step 3: Clean Up the Mesh

Apply cleanup operations. All methods are chainable and opt-in.

In [ ]:
# Each operation is opt-in. Chain what you need:
mesh.merge_seams(threshold=0.001)     # Heal tile boundary seams
mesh.remove_duplicates()               # Remove duplicate vertices
mesh.fix_normals()                     # Recalculate normals
mesh.remove_artifacts(threshold=0.1)   # Remove floating fragments
mesh.decimate(ratio=0.5)              # Reduce to 50% polygons

mesh.info()